In [ ]:
%load_ext autoreload
%autoreload 2

import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
import math
from scipy.io import loadmat

from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference
from jdcoot.utils import xcolumns, continuous_classifiers, continuous_accuracy
from sklearn.model_selection import train_test_split

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# les donnees caffeNet GoogleNet

In [ ]:
featuresToUse = ["CaffeNet4096", "GoogleNet1024"] 
#featuresToUse = ["CaffeNet4096", "CaffeNet4096"] 
sourceDomainName = ['caltech10'] #['caltech10','amazon','webcam']
targetDomainName = ['caltech10'] #['caltech10','amazon','webcam']

min_max_scaler = sklearn.preprocessing.MinMaxScaler()
# Collab
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[0],
                                                 "caltech10" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
S_data = [feat, labels]
S_nClass = len(np.unique(labels)) # nb de class in source data
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[1],
                                                 "amazon" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
T_data = [feat, labels]
T_nClass = len(np.unique(labels)) # nb de class in target data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1


In [17]:
results = []
numRepetitions = 10
alpha =1.5# hyperparamètre devant la loss a été optimé
prop_target = 0.005
algo = "sinkhorn"
reg = 1


In [18]:
a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S = source.iloc[a, :].reset_index(drop=True)
T = target.iloc[b, :].reset_index(drop=True)

In [ ]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(
                S, T, S_test, T_test,algo="sinkhorn",reg=1,batch_size=20,alpha =1.5
            )

Delta: 0.05453553408535183 	  Loss: 1.8393473970662568 	 Accuracy: 0.17079530638852672
Delta: 0.042278361239779336 	  Loss: 1.7944731530648248 	 Accuracy: 0.1590612777053455
Delta: 0.03410262716994622 	  Loss: 1.7759854758522358 	 Accuracy: 0.16297262059973924
Delta: 0.028383809153818832 	  Loss: 1.7663491869016381 	 Accuracy: 0.16036505867014342
Delta: 0.02637239268143947 	  Loss: 1.7606049723063306 	 Accuracy: 0.14993481095176012
Delta: 0.025770515640946665 	  Loss: 1.7548057552958922 	 Accuracy: 0.16166883963494133
Delta: 0.022112764323919117 	  Loss: 1.7526454389898316 	 Accuracy: 0.16427640156453716
Delta: 0.019017739576254442 	  Loss: 1.7515220105255533 	 Accuracy: 0.16297262059973924
Delta: 0.020641931078377557 	  Loss: 1.7477388663732518 	 Accuracy: 0.16166883963494133
Delta: 0.018791303606261463 	  Loss: 1.7463321219554109 	 Accuracy: 0.1590612777053455
Delta: 0.017983748064326156 	  Loss: 1.7452173011063752 	 Accuracy: 0.15645371577574968
Delta: 0.018937731946484057 	  Loss: 

In [8]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_jdcoot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                alpha=alpha,
                prop_target=0.005               
            )

Delta: 0.04383717945890281 	 Loss: 1.8018819688882228 	 Accuracy: 0.23167539267015708
Delta: 0.055515640269475676 	 Loss: 1.6279092252263683 	 Accuracy: 0.23036649214659685
Delta: 0.03795958246723688 	 Loss: 1.6054616205476746 	 Accuracy: 0.23167539267015708
Delta: 0.029932273605127516 	 Loss: 1.5969041002017894 	 Accuracy: 0.22905759162303665
Delta: 0.024186490282418024 	 Loss: 1.593085284688188 	 Accuracy: 0.23298429319371727
Delta: 0.0223830758648303 	 Loss: 1.59042553165712 	 Accuracy: 0.23167539267015708
Delta: 0.018750191394769647 	 Loss: 1.5890356900987574 	 Accuracy: 0.23167539267015708
Delta: 0.01655582575288755 	 Loss: 1.5884225017925249 	 Accuracy: 0.23167539267015708
Delta: 0.015816654235646217 	 Loss: 1.587807088354376 	 Accuracy: 0.23167539267015708
Delta: 0.014126256143828144 	 Loss: 1.5874113300444308 	 Accuracy: 0.23167539267015708
Delta: 0.014536596202974464 	 Loss: 1.5870813279147353 	 Accuracy: 0.23167539267015708
Delta: 0.012339737063951323 	 Loss: 1.58679371314711

In [10]:
pure_target

np.float64(0.22774869109947643)

In [ ]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_coot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                alpha=alpha,
                prop_target=0.005               
            )

In [ ]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_reference(
                S, T, S_test, T_test,algo=algo,reg=reg,
                alpha=alpha,
                prop_target=0.005               
            )

In [5]:
pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                alpha=alpha,
                prop_source=0.5,
                prop_target=0.005               
            )

I0000 00:00:1773692877.806990 1409726 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31129 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1773692879.052294 1410121 service.cc:148] XLA service 0x787698e49a10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773692879.052354 1410121 service.cc:156]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
I0000 00:00:1773692879.073582 1410121 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773692879.160113 1410121 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Delta: 0.0442272665521862 	  Loss: 1.884906279633607 	 Accuracy: 0.25654450261780104
Delta: 0.056260074706917795 	  Loss: 1.682564342212201 	 Accuracy: 0.25392670157068065
Delta: 0.037398100099511423 	  Loss: 1.6558062441368375 	 Accuracy: 0.2591623036649215
Delta: 0.02975214049509029 	  Loss: 1.6449148212101163 	 Accuracy: 0.25654450261780104
Delta: 0.02793553990815873 	  Loss: 1.635689796917399 	 Accuracy: 0.2630890052356021
Delta: 0.02624149811351395 	  Loss: 1.6282567825806875 	 Accuracy: 0.2670157068062827
Delta: 0.02531940500216994 	  Loss: 1.6196192269741698 	 Accuracy: 0.2709424083769634
Delta: 0.024581591077897525 	  Loss: 1.6132164735506462 	 Accuracy: 0.2696335078534031
Delta: 0.022048772085402357 	  Loss: 1.608803809741957 	 Accuracy: 0.2696335078534031
Delta: 0.022396995675887127 	  Loss: 1.6030625664139515 	 Accuracy: 0.2696335078534031
Delta: 0.02170521434141691 	  Loss: 1.5996627622990558 	 Accuracy: 0.27356020942408377
Delta: 0.021896968367035474 	  Loss: 1.59650293859

In [34]:
pure_source, pure_target, test_source, test_target = discrete_partial_coot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                alpha=alpha,
                prop_source=0.5,
                prop_target=0.005               
            )

Delta:       0.0479516 	 Loss:       1.9762464
Delta:       0.0609581 	 Loss:       1.6340702
Delta:       0.0516342 	 Loss:       1.5282199
Delta:       0.0448399 	 Loss:       1.4688440
Delta:       0.0412043 	 Loss:       1.4216875
Delta:       0.0350215 	 Loss:       1.4046976
Delta:       0.0262738 	 Loss:       1.3989525
Delta:       0.0251216 	 Loss:       1.3950071
Delta:       0.0208525 	 Loss:       1.3926575
Delta:       0.0186918 	 Loss:       1.3909998
Delta:       0.0168946 	 Loss:       1.3897824
Delta:       0.0172880 	 Loss:       1.3886747
Delta:       0.0172434 	 Loss:       1.3876016
Delta:       0.0198028 	 Loss:       1.3854229
Delta:       0.0220048 	 Loss:       1.3819343
Delta:       0.0230253 	 Loss:       1.3774892
Delta:       0.0240716 	 Loss:       1.3726989
Delta:       0.0235114 	 Loss:       1.3671677
Delta:       0.0227175 	 Loss:       1.3640346
Delta:       0.0200562 	 Loss:       1.3624380
Delta:       0.0171076 	 Loss:       1.3615987
Delta:       

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Delta:       0.0341262 	 Loss:       3.1785024
Delta:       0.0231159 	 Loss:       3.1102141
Delta:       0.0200409 	 Loss:       3.1029424
Delta:       0.0178044 	 Loss:       3.1010317
Delta:       0.0164633 	 Loss:       3.1001679
Delta:       0.0144265 	 Loss:       3.0997151
Delta:       0.0129711 	 Loss:       3.0995345
Delta:       0.0114814 	 Loss:       3.0994602
Delta:       0.0106878 	 Loss:       3.0994272
Delta:       0.0093161 	 Loss:       3.0994186
Delta:       0.0092652 	 Loss:       3.0994281
Delta:       0.0084965 	 Loss:       3.0994142
Delta:       0.0073695 	 Loss:       3.0994134
Delta:       0.0080357 	 Loss:       3.0994070
Delta:       0.0075126 	 Loss:       3.0994092
Delta:       0.0072860 	 Loss:       3.0994098
Delta:       0.0068025 	 Loss:       3.0994098
converged at iter  16


In [35]:
test_target

np.float64(0.09424083769633508)

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split

import ot
from jdcoot.coot import init_matrix_np
from jdcoot.losses import loss_crossentropy2
from jdcoot.utils import xcolumns, discrete_classifiers, discrete_accuracy
from tf_keras.utils import to_categorical


def one_hot(z, nClass):
    return to_categorical(z, num_classes=nClass)


def one_cold(z_hot):
    return np.argmax(z_hot, axis=1)

In [30]:
prop_source = 0.5
prop_target = 0.005
algo = "sinkhorn"
reg = 1
batch_size = 20
alpha = 1.5

source_levels = np.sort(np.unique(source.Z))
target_levels = np.sort(np.unique(target.Z))
nClass = len(np.union1d(source_levels, target_levels))

z_source = source.Z.values
z_target = target.Z.values

n_source = len(z_source)
n_target = len(z_target)

l_source_train, l_source_test = train_test_split(
        np.arange(n_source),
        train_size=prop_source,
    )

l_target_train, l_target_test = train_test_split(
        np.arange(n_target),
        train_size=prop_target,
    )

x_source = source.loc[:, xcolumns(source)].values
x_target = target.loc[:, xcolumns(target)].values

x_source_train = x_source[l_source_train, :]
x_target_train = x_target[l_target_train, :]

z_source_train = one_hot(z_source[l_source_train], nClass).astype(np.float64)
z_target_train = one_hot(z_target[l_target_train], nClass).astype(np.float64)

x_source_test = x_source[l_source_test, :]
z_source_test = z_source[l_source_test]

x_target_test = x_target[l_target_test, :]
z_target_test = z_target[l_target_test]
clf_source, clf_target = discrete_classifiers(source, target, "relu", "softmax")

    #algo = "sinkhorn"
     #reg = 1
algo = algo
reg = reg
algo2 = "emd"
reg2 = 0
numIterBCD = 100
nb_epoch = 10
batch_size = 20

source =S
target =T
test_source = S_test
test_target = T_test

nA, dA = x_source.shape
nB, dB = x_target.shape

vA = np.ones(dA) / dA
vB = np.ones(dB) / dB
wA = np.ones(nA) / nA
wB = np.ones(nB) / nB

    # original losses
C_s, h1_s, h2_s = init_matrix_np(x_source, x_target, vA, vB)
C_v, h1_v, h2_v = init_matrix_np(x_source.T, x_target.T, wA, wB)

cost = np.inf
Gs = np.ones((nA, nB)) / (nA * nB)
Gv = np.ones((dA, dB)) / (dA * dB)

    # we train the classifier with labelled examples only
clf_source.fit(
        x_source_train,
        z_source_train,
        batch_size=batch_size,
        epochs=nb_epoch,
        verbose=0,
    )

z_source_pred = clf_source.predict(x_source, verbose=0)

z_source_pred[l_source_train, :] = (
        z_source_train  # injection of known labels in the classifier predictions
    )

   # we train the classifier with labelled examples only
clf_target.fit(
        x_target_train, z_target_train, batch_size=batch_size, epochs=nb_epoch, verbose=0
    )

z_target_pred = clf_target.predict(x_target, verbose=0)

    # injection of known labels in the classifier predictions
z_target_pred[l_target_train] = z_target_train

log_out = {}
log_out["cost"] = []

fcost = loss_crossentropy2(z_source_pred, z_target_pred)

In [ ]:
Gsold = Gs
Gvold = Gv
costold = cost
        # step 1 : samples coupling optimization
Ms = (C_s - np.dot(h1_s, Gv).dot(h2_s.T)) + alpha * fcost  # is (nA,nB)
if algo == "emd":
            Gs = ot.emd(wA, wB, Ms, numItermax=1e7)
elif algo == "sinkhorn":
            Gs = ot.sinkhorn(wA, wB, Ms, reg)

        # step 2 : features coupling optimization
Mv = C_v - np.dot(h1_v, Gs).dot(h2_v.T)  # is (dA,dB)
if algo2 == "emd":
            Gv = ot.emd(vA, vB, Mv, numItermax=1e7)
elif algo2 == "sinkhorn":
            Gv = ot.sinkhorn(vA, vB, Mv, reg2)

delta = np.linalg.norm(Gs - Gsold) + np.linalg.norm(Gv - Gvold)
cost = np.sum(Mv * Gv)

if delta < 1e-16 or np.abs(costold - cost) < 1e-7:
            print("converged at iter ", k)


        # estimated labels of XA using transport map on samples
z_source_hat = nA * Gs.dot(z_target_pred)
z_source_hat[l_source_train] = z_source_train

In [21]:
z_source_hat = nA * Gs.dot(z_target_pred)
z_source_hat .shape

(1123, 10)

In [22]:
clf_source.fit(
            x_source, z_source_hat, batch_size=batch_size, epochs=nb_epoch, verbose=0
        )
z_source_pred = clf_source.predict(x_source, verbose=0)

z_source_pred[l_source_train] = z_source_train

z_target_hat = nB * Gs.T.dot(z_source_pred)
z_target_hat[l_target_train] = z_target_train

clf_target.fit(
            x_target, z_target_hat, batch_size=batch_size, epochs=nb_epoch, verbose=0
        )
z_target_pred = clf_target.predict(x_target, verbose=0)

z_target_pred[l_target_train] = z_target_train

accuracy = np.mean(
            (one_cold(z_target_pred[l_target_test]) + min(target_levels))
            == z_target[l_target_test]
        )

In [31]:
zpred_target = one_cold(clf_target.predict(x_target_test, verbose=0)) + min(
        target_levels
    )
zpred_source = one_cold(clf_source.predict(x_source_test, verbose=0)) + min(
        source_levels
    )

perf_pure_source = discrete_accuracy(zpred_source, z_source_test)
perf_pure_target = discrete_accuracy(zpred_target, z_target_test)
zt_test = one_cold(
        clf_target.predict(test_target.loc[:, xcolumns(test_target)], verbose=0)
    ) + min(target_levels)
zs_test = one_cold(
        clf_source.predict(test_source.loc[:, xcolumns(test_source)], verbose=0)
    ) + min(source_levels)

perf_test_source = discrete_accuracy(zs_test, test_source.Z)
perf_test_target = discrete_accuracy(zt_test, test_target.Z)


In [33]:
perf_pure_target

np.float64(0.2589098532494759)

In [69]:
from sklearn.preprocessing import OneHotEncoder as onehot
from tf_keras.utils import to_categorical
batch_size = 20
prop_target = 0.005
source =S
target =T
test_source = S_test
test_target = T_test
alpha =1.5# hyperparamètre devant la loss a été optimé
prop_target = 0.005
prop_source = 0.5
algo = "sinkhorn"
reg = 1
def one_hot(z, nClass):
    return to_categorical(z, num_classes=nClass)


def one_cold(z_hot):
    return np.argmax(z_hot, axis=1)
source_levels = np.sort(np.unique(source.Z))
target_levels = np.sort(np.unique(target.Z))

nClass = len(np.union1d(source_levels, target_levels))

z_source = source.Z.values
z_target = target.Z.values

n_source = len(z_source)
n_target = len(z_target)

l_source_train, l_source_test = train_test_split(
        np.arange(n_source),
        train_size=prop_source,
    )

l_target_train, l_target_test = train_test_split(
        np.arange(n_target),
        train_size=prop_target,
    )

x_source = source.loc[:, xcolumns(source)].values
x_target = target.loc[:, xcolumns(target)].values

x_source_train = x_source[l_source_train, :]
x_target_train = x_target[l_target_train, :]

z_source_train = one_hot(z_source[l_source_train], nClass).astype(np.float64)
z_target_train = one_hot(z_target[l_target_train], nClass).astype(np.float64)

x_source_test = x_source[l_source_test, :]
z_source_test = z_source[l_source_test]

x_target_test = x_target[l_target_test, :]
z_target_test = z_target[l_target_test]

clf_source, clf_target = discrete_classifiers(source, target, "relu", "softmax")

In [72]:
#algo = "sinkhorn"
     #reg = 1
from jdcoot.coot import init_matrix_np
from jdcoot.losses import loss_crossentropy2
algo = algo
reg = reg
algo2 = "emd"
reg2 = 0
numIterBCD = 100
nb_epoch = 10
batch_size = 20

nA, dA = x_source.shape
nB, dB = x_target.shape

vA = np.ones(dA) / dA
vB = np.ones(dB) / dB
wA = np.ones(nA) / nA
wB = np.ones(nB) / nB

    # original losses
C_s, h1_s, h2_s = init_matrix_np(x_source, x_target, vA, vB)
C_v, h1_v, h2_v = init_matrix_np(x_source.T, x_target.T, wA, wB)

cost = np.inf

Gs = np.ones((nA, nB)) / (nA * nB)
Gv = np.ones((dA, dB)) / (dA * dB)

    # we train the classifier with labelled examples only
clf_source.fit(
        x_source_train,
        z_source_train,
        batch_size=batch_size,
        epochs=nb_epoch,
        verbose=0,
    )

z_source_pred = clf_source.predict(x_source, verbose=0)

z_source_pred[l_source_train, :] = (
        z_source_train  # injection of known labels in the classifier predictions
    )

    # we train the classifier with labelled examples only
clf_target.fit(
        x_target_train, z_target_train, batch_size=batch_size, epochs=nb_epoch, verbose=0
    )

z_target_pred = clf_target.predict(x_target, verbose=0)

    # injection of known labels in the classifier predictions
z_target_pred[l_target_train] = z_target_train

log_out = {}
log_out["cost"] = []

fcost = loss_crossentropy2(z_source_pred, z_target_pred)

In [64]:
test_target

np.float64(0.20915032679738563)

In [77]:
pure_source, pure_target, test_source, test_target = discrete_partial_coot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                prop_source=0.5,prop_target=prop_target
            )

Delta:       0.0479674 	 Loss:       1.9549072
Delta:       0.0615606 	 Loss:       1.5565324
Delta:       0.0539611 	 Loss:       1.4316012
Delta:       0.0392991 	 Loss:       1.4072564
Delta:       0.0348158 	 Loss:       1.3896015
Delta:       0.0335191 	 Loss:       1.3696127
Delta:       0.0347046 	 Loss:       1.3530347
Delta:       0.0292510 	 Loss:       1.3442301
Delta:       0.0251045 	 Loss:       1.3395374
Delta:       0.0206878 	 Loss:       1.3378131
Delta:       0.0181679 	 Loss:       1.3366811
Delta:       0.0139952 	 Loss:       1.3360885
Delta:       0.0135830 	 Loss:       1.3357773
Delta:       0.0138507 	 Loss:       1.3353904
Delta:       0.0142552 	 Loss:       1.3349999
Delta:       0.0144668 	 Loss:       1.3344814
Delta:       0.0156734 	 Loss:       1.3338927
Delta:       0.0155882 	 Loss:       1.3333637
Delta:       0.0143538 	 Loss:       1.3329163
Delta:       0.0156316 	 Loss:       1.3323201
Delta:       0.0133918 	 Loss:       1.3319861
Delta:       

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/backend.py:1197: RuntimeWarning: invalid value encountered in dot
  return np.dot(a, b)
/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Delta:       0.0339445 	 Loss:       2.9216015
Delta:       0.0227292 	 Loss:       2.8502939
Delta:       0.0144800 	 Loss:       2.8456455
Delta:       0.0098485 	 Loss:       2.8452380
Delta:       0.0073204 	 Loss:       2.8451369
Delta:       0.0044932 	 Loss:       2.8451322
Delta:       0.0039989 	 Loss:       2.8451317
Delta:       0.0036054 	 Loss:       2.8451314
Delta:       0.0000000 	 Loss:       2.8451314
converged at iter  8


InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [11]:
pure_source, pure_target, test_source, test_target = discrete_partial_reference(
                S, T, S_test, T_test,algo=algo,reg=reg,
                prop_source=0.5,
                prop_target=prop_target
            )

In [ ]:
test_target

np.float64(0.5602094240837696)

In [57]:
from sklearn.preprocessing import OneHotEncoder as onehot
batch_size = 20
prop_target = 0.005
source =S
target =T
test_source = S_test
test_target = T_test
source_levels = np.sort(np.unique(source.Z))
target_levels = np.sort(np.unique(target.Z))

nClass = len(np.union1d(source_levels, target_levels))
categories = [np.arange(nClass)]

enc = onehot(handle_unknown="ignore", sparse_output=False, categories=categories)

source_train, source_test = train_test_split(source, train_size=0.5)
target_train, target_test = train_test_split(target, train_size=prop_target)
target_train


,X0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X1015,X1016,X1017,X1018,X1019,X1020,X1021,X1022,X1023,Z
347,3.226796,0.000000,3.619705,0.091887,0.0,0.000000,0.001672,0.0,0.000000,0.007939,...,3.527637,0.000000,0.251337,0.000000,0.285064,0.0,0.000000,0.0,0.000000,5.0
756,2.329142,0.050401,1.775118,3.216696,0.0,4.927517,0.388101,0.0,0.064215,0.016476,...,0.026453,0.046857,0.711302,0.420250,0.205345,0.0,0.956591,0.0,0.005848,8.0
701,1.875320,0.116668,0.967097,0.021234,0.0,0.000000,0.000000,0.0,0.011634,0.000000,...,1.403997,0.000000,0.005195,0.015218,0.000000,0.0,0.000000,0.0,0.000000,6.0


In [58]:
x_source_train = source_train.loc[:, xcolumns(source)].values
z_source_train = enc.fit_transform(source_train.Z.values[:, np.newaxis])

x_target_train = target_train.loc[:, xcolumns(target)].values
z_target_train = enc.fit_transform(target_train.Z.values[:, np.newaxis])

x_source_test = source_test.loc[:, xcolumns(source)].values
z_source_test = source_test.Z.values

x_target_test = target_test.loc[:, xcolumns(target)].values
z_target_test = target_test.Z.values

x_target_train.shape

(3, 1024)

In [59]:
from jdcoot.utils import xcolumns, discrete_accuracy, discrete_classifiers

clf_source, clf_target = discrete_classifiers(source, target, "relu", "softmax")
clf_target.fit(x_target_train, z_target_train, batch_size=batch_size, epochs=10, verbose=0)
clf_source.fit(x_source_train, z_source_train, batch_size=batch_size, epochs=10, verbose=0)

z_target_pred = enc.inverse_transform(
        clf_target.predict(x_target_test, verbose=0)
    ).ravel()
z_source_pred = enc.inverse_transform(
        clf_source.predict(x_source_test, verbose=0)
    ).ravel()

perf_pure_source = discrete_accuracy(z_source_pred, z_source_test)
perf_pure_target = discrete_accuracy(z_target_pred, z_target_test)

x_test_source = test_source.loc[:, xcolumns(source)]
x_test_target = test_target.loc[:, xcolumns(target)]

z_test_source = enc.inverse_transform(
        clf_source.predict(x_test_source, verbose=0)
    ).ravel()
z_test_target = enc.inverse_transform(
        clf_target.predict(x_test_target, verbose=0)
    ).ravel()

perf_test_source = discrete_accuracy(z_test_source, test_source.Z)
perf_test_target = discrete_accuracy(z_test_target, test_target.Z)

In [60]:
perf_test_target

np.float64(0.2094240837696335)

In [5]:
for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
    b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

    S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
    T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
    S = source.iloc[a, :].reset_index(drop=True)
    T = target.iloc[b, :].reset_index(drop=True)

    # =========================================================
    # UNSUPERVISED
    # =========================================================
     # COOT
    pure_source, pure_target, test_source, test_target =  discrete_unsupervised_coot(S, T, S_test, T_test,algo=algo,reg=reg)

    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })
    
    # JDCOOT
    pure_source, pure_target, test_source, test_target = \
         discrete_unsupervised_jdcoot(S, T, S_test, T_test, algo=algo, reg=reg, alpha=alpha)

    results.append({
         "repetition": repe,
         "recoding": "jdcoot",
         "learning": "unsupervised",
         "prop_source": 1,
         "prop_target": 0,
         "pure_source": pure_source,
         "test_source": test_source,
         "pure_target": pure_target,
         "test_target": test_target,
     })

    for prop_target in prop_target_values_s:

        # JDCOOT
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_jdcoot(
                S, T, S_test, T_test,
                alpha=alpha,
                prop_target=prop_target,
                algo=algo,
                reg=reg
            )

        results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # COOT
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_coot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "coot",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # Reference
        pure_source, pure_target, test_source, test_target = discrete_semisupervised_reference(
                S, T, S_test, T_test,
                prop_target=prop_target,algo=algo,reg=reg
            )

        results.append({
            "repetition": repe,
            "recoding": "reference",
            "learning": "semisupervised",
            "prop_source": 1,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

    # =========================================================
    # PARTIAL
    # =========================================================
    for prop_target in prop_target_values_p:

        # JDCOOT
        pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                alpha=alpha,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "jdcoot",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # COOT
        pure_source, pure_target, test_source, test_target = discrete_partial_coot(
                S, T, S_test, T_test,algo=algo,reg=reg,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "coot",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })

        # Reference
        pure_source, pure_target, test_source, test_target = discrete_partial_reference(
                S, T, S_test, T_test,algo=algo,reg=reg,
                prop_source=0.5,
                prop_target=prop_target
            )

        results.append({
            "repetition": repe,
            "recoding": "reference",
            "learning": "partial",
            "prop_source": 0.5,
            "prop_target": prop_target,
            "pure_source": pure_source,
            "test_source": test_source,
            "pure_target": pure_target,
            "test_target": test_target,
        })    

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

df_summary

df_summary.to_excel("results_mean.xlsx", index=False)
  

num repe : 1
Delta:       0.0156342 	 Loss:       2.1898716
Delta:       0.0208300 	 Loss:       2.1531316
Delta:       0.0182993 	 Loss:       2.1062000
Delta:       0.0143183 	 Loss:       2.0746391
Delta:       0.0101736 	 Loss:       2.0662397
Delta:       0.0076357 	 Loss:       2.0647772
Delta:       0.0073698 	 Loss:       2.0641208
Delta:       0.0095221 	 Loss:       2.0626732
Delta:       0.0107479 	 Loss:       2.0579851
Delta:       0.0093393 	 Loss:       2.0542179
Delta:       0.0067933 	 Loss:       2.0525879
Delta:       0.0041581 	 Loss:       2.0523363
Delta:       0.0038087 	 Loss:       2.0522088
Delta:       0.0013010 	 Loss:       2.0521508
Delta:       0.0014675 	 Loss:       2.0521515
Delta:       0.0000027 	 Loss:       2.0521512
Delta:       0.0000000 	 Loss:       2.0521512
converged at iter  16


I0000 00:00:1772268289.435497 3132503 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31129 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1772268290.478646 3133600 service.cc:148] XLA service 0x7866d9305440 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772268290.478695 3133600 service.cc:156]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
I0000 00:00:1772268290.497161 3133600 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772268290.582352 3133600 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Delta: 0.020996047184738794 	  Loss: 2.152773504142843 	 Accuracy: 0.18252933507170796
Delta: 0.018792853172571893 	  Loss: 2.063891267139697 	 Accuracy: 0.28552803129074317
Delta: 0.01669052786618614 	  Loss: 1.90707727534775 	 Accuracy: 0.30247718383311606
Delta: 0.014947723229737763 	  Loss: 1.7554657169813415 	 Accuracy: 0.2777053455019557
Delta: 0.012631281643225713 	  Loss: 1.6650440180687684 	 Accuracy: 0.2790091264667536
Delta: 0.011185817967914692 	  Loss: 1.6188625409133395 	 Accuracy: 0.28292046936114734
Delta: 0.009670421949668108 	  Loss: 1.596035899596986 	 Accuracy: 0.2894393741851369
Delta: 0.008305851206778358 	  Loss: 1.5848711222985725 	 Accuracy: 0.28552803129074317
Delta: 0.007586077124993016 	  Loss: 1.5783922549088483 	 Accuracy: 0.29986962190352023


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.006502155065100796 	  Loss: 1.5755764160556034 	 Accuracy: 0.31029986962190353
Delta: 0.006501988065390927 	  Loss: 1.5737413773666553 	 Accuracy: 0.3050847457627119
Delta: 0.003665228093089218 	  Loss: 1.573252891452163 	 Accuracy: 0.3050847457627119
Delta: 0.004760675708414077 	  Loss: 1.5730101632941702 	 Accuracy: 0.3050847457627119
Delta: 0.0025152145749396315 	  Loss: 1.5726388948500918 	 Accuracy: 0.3050847457627119
Delta: 0.0030677466573256476 	  Loss: 1.5726208908093464 	 Accuracy: 0.3050847457627119
Delta: 0.0028888098383902452 	  Loss: 1.5723986347564982 	 Accuracy: 0.3050847457627119
Delta: 0.0005120200383004244 	  Loss: 1.572406464929189 	 Accuracy: 0.3050847457627119
Delta: 0.0025240901477323005 	  Loss: 1.5723857840932247 	 Accuracy: 0.3050847457627119
Delta: 0.0012899934164273348 	  Loss: 1.5724185913400657 	 Accuracy: 0.3050847457627119
Delta: 0.0008160867488172346 	  Loss: 1.5723351724664298 	 Accuracy: 0.3076923076923077
Delta: 0.001369778596900384 	  Loss: 

...

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.004854014746171697 	 Loss: 1.5356051933581867 	 Accuracy: 0.8477366255144033
Delta: 0.004757824338656748 	 Loss: 1.5357031469413756 	 Accuracy: 0.8491083676268861
Delta: 0.003709637315386927 	 Loss: 1.5356368391650204 	 Accuracy: 0.850480109739369
Delta: 0.0020409776017180146 	 Loss: 1.5358441662601314 	 Accuracy: 0.850480109739369
Delta: 0.0037538323029430386 	 Loss: 1.5358093314629868 	 Accuracy: 0.8477366255144033
Delta: 0.0014817617293149212 	 Loss: 1.5358550295310298 	 Accuracy: 0.8491083676268861
Delta: 0.002304659664941159 	 Loss: 1.535786838976604 	 Accuracy: 0.8491083676268861
Delta: 0.0017876326748458697 	 Loss: 1.5356971973192057 	 Accuracy: 0.8573388203017832
Delta: 0.0037760905563686575 	 Loss: 1.5353856371652022 	 Accuracy: 0.850480109739369
Delta: 0.0014451226635680393 	 Loss: 1.535440653566987 	 Accuracy: 0.8491083676268861
Delta: 0.001104397592097245 	 Loss: 1.5356086045303403 	 Accuracy: 0.8518518518518519
Delta: 5.017585983622265e-05 	 Loss: 1.53527868504538

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.004649879075309851 	 Loss: 1.5274076762505968 	 Accuracy: 0.874095513748191
Delta: 0.004913831413216494 	 Loss: 1.5271973920705009 	 Accuracy: 0.8712011577424024
Delta: 0.004583021685523163 	 Loss: 1.5269997143464886 	 Accuracy: 0.869753979739508
Delta: 0.0018247908273537998 	 Loss: 1.5269980588572871 	 Accuracy: 0.8683068017366136
Delta: 0.0007692477557217745 	 Loss: 1.5268906248170588 	 Accuracy: 0.8683068017366136
Delta: 0.0019479872428666246 	 Loss: 1.5271231494131907 	 Accuracy: 0.8668596237337193
Delta: 0.003172211196973857 	 Loss: 1.527236680063962 	 Accuracy: 0.8668596237337193
Delta: 0.0036128828623614214 	 Loss: 1.52742506366116 	 Accuracy: 0.8668596237337193
Delta: 0.0005570042541673546 	 Loss: 1.527081310417558 	 Accuracy: 0.8683068017366136
Delta: 1.5265923034363353e-05 	 Loss: 1.52705398240037 	 Accuracy: 0.8668596237337193
Delta: 0.0010781371449413077 	 Loss: 1.52722489023756 	 Accuracy: 0.8668596237337193
Delta: 4.6533502305763976e-05 	 Loss: 1.5272452033139414

KeyboardInterrupt: 

10

In [12]:
perf_test_target

np.float64(0.9322916666666666)

In [10]:
a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S = source.iloc[a, :].reset_index(drop=True)
T = target.iloc[b, :].reset_index(drop=True)

import numpy as np
from sklearn.preprocessing import OneHotEncoder as onehot
from sklearn.model_selection import train_test_split
from jdcoot.utils import xcolumns, discrete_classifiers, discrete_accuracy
prop_source = 0.8
prop_target = 0.8
source_levels = np.sort(np.unique(S.Z))
target_levels = np.sort(np.unique(T.Z))

nClass = len(np.union1d(source_levels, target_levels))
categories = [np.arange(nClass)]

enc = onehot(handle_unknown="ignore", sparse_output=False, categories=categories)



x_source_train = S.loc[:, xcolumns(S)].values
z_source_train = enc.fit_transform(S.Z.values[:, np.newaxis])

x_target_train = T.loc[:, xcolumns(T)].values
z_target_train = enc.fit_transform(T.Z.values[:, np.newaxis])

x_source_test = S_test.loc[:, xcolumns(S_test)].values
z_source_test = S_test.Z.values

x_target_test = T_test.loc[:, xcolumns(T_test)].values
z_target_test = T_test.Z.values

clf_source, clf_target = discrete_classifiers(source, target, "relu", "softmax")

clf_target.fit(x_target_train, z_target_train, batch_size=20, epochs=20, verbose=0)
clf_source.fit(x_source_train, z_source_train, batch_size=20, epochs=20, verbose=0)

z_target_pred = enc.inverse_transform(
        clf_target.predict(x_target_test, verbose=0)
    ).ravel()
z_source_pred = enc.inverse_transform(
        clf_source.predict(x_source_test, verbose=0)
    ).ravel()

perf_test_source = discrete_accuracy(z_source_pred,S_test.Z)
perf_test_target = discrete_accuracy(z_target_pred, T_test.Z)

In [6]:
perf_test_target

np.float64(0.9581151832460733)

In [ ]:
import numpy as np
# Exemple de valeurs à tester pour alpha
alpha_values = np.linspace(1, 3, 21)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique
results = []
numRepetitions = 10
for repe in range(numRepetitions):
   
        a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
        b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

        S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
        T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
        S = source.iloc[a, :].reset_index(drop=True)
        T = target.iloc[b, :].reset_index(drop=True)
        for a in alpha_values:
            pure_source, pure_target, test_source, test_target = \
            discrete_unsupervised_jdcoot(S, T, S_test, T_test,alpha=a)

            score = test_target  
    
            if score > best_score:
                best_score = score
                best_alpha = a

        results.append({
         "repetition": repe,
         "recoding": "jdcoot",
         "learning": "unsupervised",
         "alpha": best_alpha,
     })

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "alpha"],
        as_index=False
    )
    .agg(
        alpha_mean=("alpha", "mean"),
    )
)

df_summary

df_summary.to_excel("results_mean.xlsx", index=False)


Delta: 0.02092633484461369 	  Loss: 2.14224528364411 	 Accuracy: 0.18252933507170796
Delta: 0.018176953250223424 	  Loss: 2.076236270607468 	 Accuracy: 0.28292046936114734
Delta: 0.016913402047423795 	  Loss: 1.9716716026534993 	 Accuracy: 0.40547588005215124
Delta: 0.014520431794906617 	  Loss: 1.8454590126498591 	 Accuracy: 0.5097783572359843
Delta: 0.011881452125277088 	  Loss: 1.7546242454622245 	 Accuracy: 0.5228161668839635
Delta: 0.009674465390928417 	  Loss: 1.698684494847213 	 Accuracy: 0.5202086049543677
Delta: 0.008769443620514402 	  Loss: 1.662466722491427 	 Accuracy: 0.5215123859191656
Delta: 0.0073013686771785545 	  Loss: 1.638404211386383 	 Accuracy: 0.5228161668839635
Delta: 0.007038005114058835 	  Loss: 1.6199267417866 	 Accuracy: 0.5228161668839635
Delta: 0.007591871689274479 	  Loss: 1.6066873126727785 	 Accuracy: 0.5176010430247718
Delta: 0.006789434164771827 	  Loss: 1.5963459873051473 	 Accuracy: 0.5241199478487614
Delta: 0.006842256866761354 	  Loss: 1.5880158227

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.004268351425853599 	  Loss: 1.5302416076309593 	 Accuracy: 0.5971316818774446
Delta: 0.002468600151216607 	  Loss: 1.5298568680534534 	 Accuracy: 0.6010430247718384
Delta: 0.003409394980149225 	  Loss: 1.5294051191112767 	 Accuracy: 0.6036505867014341
Delta: 0.0026056418154393524 	  Loss: 1.5290989163623905 	 Accuracy: 0.6036505867014341
Delta: 0.004450593221749945 	  Loss: 1.5287926472316449 	 Accuracy: 0.6010430247718384
Delta: 0.0019982357466058792 	  Loss: 1.5285084229842665 	 Accuracy: 0.6010430247718384
Delta: 0.003024546015134233 	  Loss: 1.5281522435370294 	 Accuracy: 0.6036505867014341
Delta: 0.0031814445326312723 	  Loss: 1.5279872360789952 	 Accuracy: 0.6023468057366362
Delta: 0.002331933267503773 	  Loss: 1.5276707170414545 	 Accuracy: 0.5997392438070405
Delta: 0.003337752432730538 	  Loss: 1.5275016550320366 	 Accuracy: 0.6036505867014341
Delta: 0.00247531803093028 	  Loss: 1.52738016471346 	 Accuracy: 0.6023468057366362
Delta: 0.003883076425155863 	  Loss: 1.5271

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.004250858110592078 	  Loss: 1.5375583315617698 	 Accuracy: 0.5971316818774446
Delta: 0.004925172950520025 	  Loss: 1.5366972513651223 	 Accuracy: 0.5971316818774446
Delta: 0.0025573262853966373 	  Loss: 1.536596680295323 	 Accuracy: 0.6023468057366362
Delta: 0.0030462749842994933 	  Loss: 1.5363899627465376 	 Accuracy: 0.6010430247718384
Delta: 0.004018761166484025 	  Loss: 1.53628394740011 	 Accuracy: 0.6010430247718384
Delta: 0.0035866075779905095 	  Loss: 1.5358793178845394 	 Accuracy: 0.6010430247718384
Delta: 0.0028464444850449413 	  Loss: 1.5356471662770999 	 Accuracy: 0.5958279009126467
Delta: 0.0030939960551417058 	  Loss: 1.5355380805102017 	 Accuracy: 0.5945241199478487
Delta: 0.0014537634695905965 	  Loss: 1.535479073065265 	 Accuracy: 0.5997392438070405
Delta: 0.0026527589885764046 	  Loss: 1.535308454312641 	 Accuracy: 0.5984354628422425
Delta: 0.00432571082097549 	  Loss: 1.5351977496988485 	 Accuracy: 0.6010430247718384
Delta: 0.003790032444591693 	  Loss: 1.534

...

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.011304516150701457 	  Loss: 1.6597309062507593 	 Accuracy: 0.5032594524119948
Delta: 0.008407904680951669 	  Loss: 1.6551897949919803 	 Accuracy: 0.5045632333767927
Delta: 0.005320573827949609 	  Loss: 1.6544238260763686 	 Accuracy: 0.4980443285528031
Delta: 0.0040515251132920435 	  Loss: 1.6541218769467985 	 Accuracy: 0.5032594524119948
Delta: 0.0033717341739909817 	  Loss: 1.6541806748343675 	 Accuracy: 0.5032594524119948
Delta: 0.0038704014856199478 	  Loss: 1.6541828455603147 	 Accuracy: 0.5045632333767927
Delta: 0.0032225236781563142 	  Loss: 1.654013978138905 	 Accuracy: 0.5019556714471969
Delta: 0.0012691913624802407 	  Loss: 1.6538348029888854 	 Accuracy: 0.5058670143415906
Delta: 4.352510831637674e-05 	  Loss: 1.6537836305160338 	 Accuracy: 0.49934810951760106
Delta: 2.9845516617307103e-05 	  Loss: 1.653625965261495 	 Accuracy: 0.5045632333767927
Delta: 0.001325308195433357 	  Loss: 1.653949589777899 	 Accuracy: 0.49674054758800523
Delta: 0.004183840035226078 	  Loss:

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.007877896168171906 	  Loss: 1.617434665995532 	 Accuracy: 0.3963494132985658
Delta: 0.006260142914317923 	  Loss: 1.6159225212093753 	 Accuracy: 0.3963494132985658
Delta: 0.00442915526269464 	  Loss: 1.6154784720566615 	 Accuracy: 0.3924380704041721
Delta: 0.003283067971059584 	  Loss: 1.6157770781232301 	 Accuracy: 0.39504563233376794
Delta: 0.0034389915369266036 	  Loss: 1.6158466804580471 	 Accuracy: 0.39374185136897
Delta: 0.0015444669274346863 	  Loss: 1.6158155611839304 	 Accuracy: 0.39374185136897
Delta: 0.003573855018048309 	  Loss: 1.616013966368004 	 Accuracy: 0.39374185136897
Delta: 0.0013560211520524592 	  Loss: 1.6160902259896348 	 Accuracy: 0.39374185136897
Delta: 3.3647097517548485e-05 	  Loss: 1.6160985956042886 	 Accuracy: 0.39374185136897
Delta: 0.001447026010975401 	  Loss: 1.6160922551538244 	 Accuracy: 0.39374185136897
Delta: 0.0013692448984229208 	  Loss: 1.6160770352260565 	 Accuracy: 0.39374185136897
Delta: 0.0007821985160665661 	  Loss: 1.6158241199311

,recoding,learning,alpha,alpha_mean
0,jdcoot,unsupervised,1.0,1.0
1,jdcoot,unsupervised,1.9,1.9


In [6]:
df_summary.to_excel("results_mean.xlsx", index=False)

In [8]:
df_results 

,repetition,recoding,learning,alpha
0,0,jdcoot,unsupervised,1.9
1,1,jdcoot,unsupervised,1.9
2,2,jdcoot,unsupervised,1.9
3,3,jdcoot,unsupervised,1.9
4,4,jdcoot,unsupervised,1.9
5,5,jdcoot,unsupervised,1.9
6,6,jdcoot,unsupervised,1.9
7,7,jdcoot,unsupervised,1.0
8,8,jdcoot,unsupervised,1.0
9,9,jdcoot,unsupervised,1.0
